<a href="https://colab.research.google.com/github/qaz9391/LINE-BOT-B12090026/blob/main/0702_colab_line_bot_with_gemini_stateful.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.5/819.5 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 6.2 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051


In [ ]:
import os
from pyngrok import ngrok

In [ ]:
ngrok.kill()

In [ ]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

Ngrok URL: https://blazer-broadly-harpist.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://blazer-broadly-harpist.ngrok-free.dev


True

In [ ]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

# === 初始化 Google Gemini ===
client = genai.Client(api_key=gemini_api_key)

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=GenerateContentConfig(
        response_modalities=["TEXT"],#就是在這在程式中註解為何回應是stateful#
    )
)

In [ ]:
def stateful_query(payload):
    response = chat.send_message(message=payload)
    return response.text

In [ ]:
result = stateful_query("簡介明新科技大學")
print(result)

明新科技大學（Ming Hsin University of Science and Technology, MHUST），簡稱「明新科大」，是一所位於台灣新竹縣新豐鄉的私立科技大學。

以下是其簡要介紹：

1.  **創校歷史與沿革：**
    *   創立於**1966年**，前身為「明新工業專科學校」。
    *   1997年改制為「明新技術學院」。
    *   **2002年**升格並更名為「明新科技大學」。
    *   歷經數十年的發展，已從專科學校逐步升格為綜合性科技大學，並持續深化技職教育特色。

2.  **教育理念與特色：**
    *   **校訓：** 誠、信、勤、毅。
    *   **核心理念：** 學校秉持「**務實致用**」的教育理念，強調理論與實務並重，致力於培養具備專業技能、創新能力和國際視野的優質人才。
    *   **產學合作：** 與鄰近新竹科學園區及各行各業建立了緊密的產學合作關係，提供學生豐富的實習機會、企業參訪及專題研究，提升學生的職場競爭力。

3.  **學院與系所：**
    *   學校設有多個學院，涵蓋工程、管理、服務事業、設計等多元領域，提供學士、碩士等學位課程。
    *   **主要學院包括：**
        *   **工程學院：** 如機械工程系、電子工程系、資訊工程系、土木工程系等。
        *   **管理學院：** 如企業管理系、資訊管理系、國際商務系、行銷與流通管理系等。
        *   **服務事業學院：** 如觀光事業系、旅館事業管理系、運動管理系、幼兒保育系等。
        *   **設計學院：** 如多媒體與遊戲設計系、時尚造型設計系、空間設計系等。
    *   系所設置緊密結合產業需求與未來發展趨勢。

4.  **校園環境與資源：**
    *   校園環境優美，佔地廣闊，各項設施完善。
    *   擁有現代化的教學大樓、專業實驗室、智慧教室、圖書館、體育館、學生宿舍、藝文中心等多功能空間，為學生提供良好的學習與生活環境。

5.  **重要成就與展望：**
    *   明新科大在教學品質與學生就業方面表現良好，曾多次獲得教育部教學卓越計畫、技職再造計畫等各項補助與肯定。
    *   學校積極推動國際化，與多國

In [ ]:
result2 = stateful_query("校長是誰？")
print(result2)

明新科技大學現任校長是 **劉國偉 教授**。


In [ ]:
from flask import Flask, request, abort

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
)

app = Flask(__name__)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)


@app.route("/", methods=['POST'])
def callback():
    # get X-Line-Signature header value
    signature = request.headers['X-Line-Signature']

    # get request body as text
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    # handle webhook body
    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature. Please check your channel access token/channel secret.")
        abort(400)

    return 'OK'


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    text = event.message.text
    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)
        if text.startswith('AI '):
            prompt = text[3:]
            reply_text = stateful_query(prompt)
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=reply_text)]
                )
            )

        else:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=event.message.text),
                        TextMessage(text=event.message.text)]
                )
            )

if __name__ == "__main__":
    app.run(port=port)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit


BODY:  {"destination":"Ue9bc54ad0cac458e5ed676a3e8751c97","events":[{"type":"message","message":{"type":"text","id":"616774759795654713","quoteToken":"1Vj03n60L6MS72F0Avtd6bQNbb-qu8bYC8yqF5oC3bJX3KRF8cI618Z5MVi7dUOp_DhRvLhf46aBxa1ofaaH8jKDsb0Edw_puY6z6ObdwDcWW1Mhb01MV-o9-CNeT6lHcApXhkGrOxhz3hQbyBP6SQ","markAsReadToken":"EtUPm8Dg3gqGA632xe0ZF_TjU702O7bbpGfNEEDtjn94tCXFx4qIyg-MApnY6n1SCeYgMdGySHZissiICdm_3c08Q98BnwxXE7b_LJn06AOpa6X10UxVnRSYLlH4GNtGiteRAFbF4VSx1KyxBSzM-Mrjc2lhjBYmnAC1Y6hq69IeWgNlQvRJh2hwx5VOVfw18V7u4qomtkxgNx83l9NBNw","text":"AI 介紹明新科大，20字內"},"webhookEventId":"01KT5RS7SA4FNG8RMEHKA916QK","deliveryContext":{"isRedelivery":false},"timestamp":1780457708846,"source":{"type":"user","userId":"U4a7effcc683b4b1daf00f53459845948"},"replyToken":"154c413ad1634fc5beb9fff9b7760273","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [03/Jun/2026 03:35:11] "POST / HTTP/1.1" 200 -


BODY:  {"destination":"Ue9bc54ad0cac458e5ed676a3e8751c97","events":[{"type":"message","message":{"type":"text","id":"616774781638803458","quoteToken":"nUWWtt1OpSrgtaNlSX-2aLydCrd2Cka6Tu5x3RRJzxVpI0cYjjAdo9QhgvTT5NOsfLMQNVFjrDNcZKW3IzATkWygLcHRrG4WZ3qIB97pBEPDbIkcj0N98pYadN2CgFloAdSvNwRZePgB1SQayTPLcw","markAsReadToken":"Iwp8FcExthpzuk79xQ4a3V_wffur11eEKqOq26vuRDLdfeT7C_00OgcKZZSsm0j9ZjBLJMFe1I8GepQfzF4BmAZJyA2R3uy7lCJ2JJH_un4wBnS7OUZA8hXC-5hRN0ddmpt8ViGN0WOg3LVlnIYItzafoqVf46g-Ha8THR0tYXsKL2D8I6aU90JkoHRUsLMk3n68IX6lsSnnCwZGbFCteA","text":"AI 校長是誰"},"webhookEventId":"01KT5RSMA2TD5BETDMR9FY3STK","deliveryContext":{"isRedelivery":false},"timestamp":1780457721674,"source":{"type":"user","userId":"U4a7effcc683b4b1daf00f53459845948"},"replyToken":"530a30bfd8874d1497a1c7eba2f8b44e","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [03/Jun/2026 03:35:23] "POST / HTTP/1.1" 200 -
